From an Google Search 'Word2Vec' (https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfDjGCbGLkqwvITjmZJS3lyGsQh-otF9QlZqdVr5gU5RMD2IR2k1L8GgtstNNOkW8V58BHVtijYOvkBDi7NBM5A_oMG-Zf0baGofEpssl-8tXTTBHYJ-zAyQf1SsQaucVv5i4H8Y9dLE6rxIA0H8BWHQDetvUAovqUuPvmgMDxNYH-jXqmnl36Mo1P1SpuyQKJJ34T4c3k2XTIGzOnTpSBNSxkUATZYU0fZjdRK4CXHm30zQHWnNHqBKrc5SRmytvIXI2UnV4_CWRQ&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50)



#### Install Libraries

In [2]:
# Step 1: Install the Necessary Libraries

!pip install transformers datasets scikit-learn pandas


#### Load a Pre-trained Medical BERT Model:

In [3]:
# Step 2: Load a Pre-trained Medical BERT Model:

import torch
from transformers import AutoTokenizer, AutoModel

# Check that PyTorch sees your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the medical BERT tokenizer and model (BioBERT)
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

print("✅ Medical BERT model successfully loaded onto your GPU!")


Using device: cpu


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Medical BERT model successfully loaded onto your GPU!


#### Load/Create Datasets

##### Try this with 500 realistic records from Hugging Face, non-duplicated:

#### **Use a non-duplicated set of 500 cells.**

In [4]:
import pandas as pd
from datasets import load_dataset

print("🔄 Streaming real unique medical records from Hugging Face...")
raw_dataset = load_dataset("NickyNicky/medical_mtsamples", split="train")
all_records = pd.DataFrame(raw_dataset)

# 1. Standardize columns and drop missing data
all_records = all_records.rename(columns={'medical_specialty': 'specialty', 'transcription': 'clinical_notes'})
all_records = all_records[['specialty', 'clinical_notes']].dropna()
all_records['specialty'] = all_records['specialty'].str.strip()
all_records['clinical_notes'] = all_records['clinical_notes'].str.strip()

# 2. Filter for our target specialties
target_patterns = 'Cardiology|Orthopedics|Pediatrics|Neurology|Gastroenterology'
filtered_df = all_records[all_records['specialty'].str.contains(target_patterns, case=False, na=False)].copy()

# 3. CRUCIAL CHANGE: Drop all duplicate text records entirely to prevent artificial bias
filtered_df = filtered_df.drop_duplicates(subset=['clinical_notes'])

# 4. Ingest ALL available unique records instead of duplicating to 990
df_real = filtered_df.copy().reset_index(drop=True)
df_real['encounter_id'] = range(1001, 1001 + len(df_real))

print(f"✅ Baseline established with {len(df_real)} completely UNIQUE medical records.")

# 5. Inject your 10 target anomalies
anomalies_pool = [
    {"encounter_id": 2001, "specialty": "Neurology", "clinical_notes": "Patient presents to emergency department with sudden crushing retrosternal chest pain radiating to left jaw, accompanied by profuse sweating and dyspnea. Initial EKG reveals acute ST-segment elevation. Prepared for immediate cardiac catheterization."},
    {"encounter_id": 2002, "specialty": "Pediatrics", "clinical_notes": "An 84-year-old resident of an assisted living facility is evaluated for advanced cognitive decline, severe resting tremors in the right hand, and shuffling gait mechanics. Initiating trial of Carbidopa-Levodopa for Parkinsonian features."},
    {"encounter_id": 2003, "specialty": "Orthopedics", "clinical_notes": "Chief complaint centers on burning epigastric pain radiating to the chest, significantly worse postprandially and when recumbent. Patient reports passing dark, tarry stools for two days. Urgently scheduling an upper endoscopy (EGD)."},
    {"encounter_id": 2004, "specialty": "Cardiology", "clinical_notes": "A 4-year-old male child is brought in by his mother for a routine pediatric checkup and preschool clearance physical. Growth percentiles are tracking normally along the 70th line. Administered MMR and Varicella booster vaccines."},
    {"encounter_id": 2005, "specialty": "Gastroenterology", "clinical_notes": "The patient is a 22-year-old varsity athlete who experienced an inversion injury to the right ankle during a basketball match. Marked localized edema and ecchymosis noted. Radiograph indicates an acute non-displaced distal fibula fracture."},
    {"encounter_id": 2006, "specialty": "Neurology", "clinical_notes": "Patient presents with persistent watery diarrhea, severe lower abdominal cramping, and low-grade pyrexia following a recent course of Clindamycin. Stool assay is ordered to evaluate for Clostridioides difficile infection."},
    {"encounter_id": 2007, "specialty": "Pediatrics", "clinical_notes": "A 67-year-old postmenopausal female presents with a bone mineral density T-score of -2.8 via DEXA scan, indicating severe systemic osteoporosis. Initiating monthly oral Bisphosphonate therapy and calcium supplementation."},
    {"encounter_id": 2008, "specialty": "Cardiology", "clinical_notes": "Evaluation of a 45-year-old female complaining of a worsening throbbing unilateral headache accompanied by severe photophobia, phonophobia, and visual aura. Initiating prophylactic treatment with Topiramate."},
    {"encounter_id": 2009, "specialty": "Gastroenterology", "clinical_notes": "Patient displays an acute onset of left-sided facial droop, slurred speech, and pronator drift of the left upper extremity. Transferred emergently to the neuro-ICU for acute ischemic stroke evaluation and potential tPA administration."},
    {"encounter_id": 2010, "specialty": "Orthopedics", "clinical_notes": "Patient presents with progressive exertional dyspnea, orthopnea, and 3+ pitting edema of the lower extremities. Transthoracic echocardiogram demonstrates an ejection fraction reduced to 25%, consistent with congestive heart failure."}
]

df = pd.concat([df_real, pd.DataFrame(anomalies_pool)], ignore_index=True)
df = df.sort_values(by="encounter_id").reset_index(drop=True)
print(f"📊 Global Dataset Finalized! Total records: {len(df)}")


🔄 Streaming real unique medical records from Hugging Face...


mtsamples (1).csv: reconstructing file:   0%|          |  0.00B / 17.0MB            

mtsamples (1).csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4999 [00:00<?, ? examples/s]

✅ Baseline established with 510 completely UNIQUE medical records.
📊 Global Dataset Finalized! Total records: 520


#### Set up a pre-BERT Filter

In [6]:
# Pre-BERT Filter


import pandas as pd
import numpy as np

def run_pre_embedding_audit(dataframe):
    print("================================================================================")
    print("🛡️ RUNNING PRE-EMBEDDING DEFENSE FILTERS")
    print("================================================================================")

    # Create copies to prevent modifying the original data securely
    working_df = dataframe.copy()

    # Initialize lists to hold our filtered-out queues
    queue_a_insufficient = []
    queue_b_incomplete = []
    queue_pass_to_ml = []

    for index, row in working_df.iterrows():
        notes = str(row['clinical_notes']).strip()
        word_count = len(notes.split())
        char_count = len(notes)

        # ----------------------------------------------------------------------
        # LAYER 1: Word Count & Absolute Character Gatekeeper
        # ----------------------------------------------------------------------
        if char_count < 20 or word_count < 15:
            queue_a_insufficient.append({
                'encounter_id': row['encounter_id'],
                'specialty': row['specialty'],
                'issue': f"Severe Under-Documentation (Words: {word_count}, Chars: {char_count})",
                'text_snippet': notes[:80]
            })
            continue

        # ----------------------------------------------------------------------
        # LAYER 2: Sentence Incompleteness Parser
        # ----------------------------------------------------------------------
        last_char = notes[-1] if len(notes) > 0 else ""
        if last_char not in ['.', '!', '?']:
            queue_b_incomplete.append({
                'encounter_id': row['encounter_id'],
                'specialty': row['specialty'],
                'issue': f"Mid-Sentence Save Drop (Terminal character: '{last_char}')",
                'text_snippet': notes[-80:]
            })
            continue

        # If it passes both defense shields, it is safe to send to BioBERT
        queue_pass_to_ml.append(row)

    # Convert our results back into clean DataFrames for down-stream processing
    df_queue_a = pd.DataFrame(queue_a_insufficient)
    df_queue_b = pd.DataFrame(queue_b_incomplete)
    df_clean_ml = pd.DataFrame(queue_pass_to_ml).reset_index(drop=True)

    # Print out our audit discovery logs
    print(f"✅ Filter Complete! Clean records passed to ML Pipeline: {len(df_clean_ml)}")
    print(f"📥 ROUTED TO [QUEUE A - UNDER-DOCUMENTATION FILTERS]: {len(df_queue_a)}")
    print(f"📥 ROUTED TO [QUEUE B - MID-SENTENCE DROP FILTERS]   : {len(df_queue_b)}\n")

    # FIX: Loop through the raw dictionary list instead of the DataFrame columns
    if len(queue_a_insufficient) > 0:
        print("⚠️ SAMPLE OF BLANK / SEVERELY UNDER-DOCUMENTED RECORDS FOUND:")
        for item in queue_a_insufficient[:3]:
            print(f"  • ID: {item['encounter_id']} | Label: {item['specialty']} | Reason: {item['issue']}")
            print(f"    Text: \"{item['text_snippet']}\"")
        print()

    if len(queue_b_incomplete) > 0:
        print("⚠️ SAMPLE OF TRANSCRIPTION SAVES HALTED MID-SENTENCE:")
        for item in queue_b_incomplete[:3]:
            print(f"  • ID: {item['encounter_id']} | Label: {item['specialty']} | Reason: {item['issue']}")
            print(f"    Trailing End: \"...{item['text_snippet']}\"")
        print("================================================================================\n")

    return df_clean_ml, df_queue_a, df_queue_b

# Execute the pre-filters on your current main dataframe
df_clean, queue_a, queue_b = run_pre_embedding_audit(df)



🛡️ RUNNING PRE-EMBEDDING DEFENSE FILTERS
✅ Filter Complete! Clean records passed to ML Pipeline: 462
📥 ROUTED TO [QUEUE A - UNDER-DOCUMENTATION FILTERS]: 4
📥 ROUTED TO [QUEUE B - MID-SENTENCE DROP FILTERS]   : 54

⚠️ SAMPLE OF BLANK / SEVERELY UNDER-DOCUMENTED RECORDS FOUND:
  • ID: 1332 | Label: Gastroenterology | Reason: Severe Under-Documentation (Words: 5, Chars: 40)
    Text: "PREOPERATIVE DIAGNOSIS: , Biliary colic."
  • ID: 1336 | Label: Gastroenterology | Reason: Severe Under-Documentation (Words: 5, Chars: 71)
    Text: "PREOPERATIVE DIAGNOSIS:,  Cholelithiasis; possible choledocholithiasis."
  • ID: 1365 | Label: Gastroenterology | Reason: Severe Under-Documentation (Words: 10, Chars: 91)
    Text: "INDICATION:  , Rectal bleeding.,PREMEDICATION:, See procedure nurse NCS form.,PR"

⚠️ SAMPLE OF TRANSCRIPTION SAVES HALTED MID-SENTENCE:
  • ID: 1026 | Label: Pediatrics - Neonatal | Reason: Mid-Sentence Save Drop (Terminal character: 'o')
    Trailing End: "...r reconstitution  S

#### List records caught by the pre-embedding audit filters.

In [16]:
print("==================================================")
print("📥 RECORDS CAUGHT BY PRE-EMBEDDING AUDIT FILTERS")
print("==================================================")

# 1. Print out the Under-Documentation Filter captures (Queue A)
print(f"⚠️ QUEUE A: Insufficient Text / Blank Records ({len(queue_a)} total)")
if len(queue_a) > 0:
    for _, item in queue_a.iterrows():
        print(f"  • ID: {item['encounter_id']} | Specialty: {item['specialty']}")
else:
    print("  No records caught.")

print("\n" + "-"*50 + "\n")

# 2. Print out the Mid-Sentence Save Drop captures (Queue B)
print(f"⚠️ QUEUE B: Mid-Sentence Save Drops ({len(queue_b)} total)")
if len(queue_b) > 0:
    for _, item in queue_b.iterrows():
        print(f"  • ID: {item['encounter_id']} | Specialty: {item['specialty']}")
else:
    print("  No records caught.")
print("==================================================")

📥 RECORDS CAUGHT BY PRE-EMBEDDING AUDIT FILTERS
⚠️ QUEUE A: Insufficient Text / Blank Records (4 total)
  • ID: 1332 | Specialty: Gastroenterology
  • ID: 1336 | Specialty: Gastroenterology
  • ID: 1365 | Specialty: Gastroenterology
  • ID: 1407 | Specialty: Gastroenterology

--------------------------------------------------

⚠️ QUEUE B: Mid-Sentence Save Drops (54 total)
  • ID: 1026 | Specialty: Pediatrics - Neonatal
  • ID: 1027 | Specialty: Pediatrics - Neonatal
  • ID: 1028 | Specialty: Pediatrics - Neonatal
  • ID: 1031 | Specialty: Pediatrics - Neonatal
  • ID: 1038 | Specialty: Pediatrics - Neonatal
  • ID: 1043 | Specialty: Pediatrics - Neonatal
  • ID: 1045 | Specialty: Pediatrics - Neonatal
  • ID: 1046 | Specialty: Pediatrics - Neonatal
  • ID: 1048 | Specialty: Pediatrics - Neonatal
  • ID: 1067 | Specialty: Pediatrics - Neonatal
  • ID: 1069 | Specialty: Pediatrics - Neonatal
  • ID: 1105 | Specialty: Neurology
  • ID: 1106 | Specialty: Neurology
  • ID: 1107 | Specialty

In [1]:
#### Process Text - this is for 500 records

In [7]:
import torch
import numpy as np
from tqdm.notebook import tqdm

# 1. Combine fields to provide full context
df['combined_text'] = "Specialty: " + df['specialty'] + " | Notes: " + df['clinical_notes']

# 2. Put the model in evaluation mode on the GPU
model.eval()

# Optimized batch extraction function
def get_bert_embeddings_batched(text_list, batch_size=16):
    all_embeddings = []

    # Process records in chunks (batches) to maximize GPU speed safely
    for i in tqdm(range(0, len(text_list), batch_size), desc="Processing Batches"):
        batch_texts = text_list[i:i+batch_size]

        # Tokenize the entire batch at once
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        # Extract features without calculating gradients (saves memory)
        with torch.no_grad():
            outputs = model(**inputs)

        # Pull vectors from GPU memory back to CPU as a NumPy matrix
        embeddings = outputs.pooler_output.cpu().numpy()
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

print("Starting BioBERT vector extraction on 1,000 records...")
# Run the batch process on our unified text column
X = get_bert_embeddings_batched(df['combined_text'].tolist(), batch_size=16)

# Save the individual vectors back into the dataframe for our downstream scripts
df['vector'] = list(X)
print(f"✅ Finished! Created vector matrix with shape: {X.shape}")


Starting BioBERT vector extraction on 1,000 records...


Processing Batches:   0%|          | 0/33 [00:00<?, ?it/s]

✅ Finished! Created vector matrix with shape: (520, 768)


In [ ]:

print(X)
type(X)



In [ ]:
#### Convert Text into Medical BERT Embeddings:

In [8]:
# Step 5: Convert Text into Medical BERT Embeddings:

import torch
import numpy as np

# Ensure your model from the previous step is active
model.eval()

def get_bert_embedding(text):
    # Tokenize text and move tensors to the GPU
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the 'Pooler Output' which represents the semantic meaning of the entire text block
    embeddings = outputs.pooler_output.cpu().numpy()
    return embeddings[0]

# Generate vectors for all clinical notes
print("Processing text through BioBERT on the GPU...")
df['vector'] = df['clinical_notes'].apply(get_bert_embedding)
print("✅ Text converted into dense math vectors successfully!")



Processing text through BioBERT on the GPU...
✅ Text converted into dense math vectors successfully!


#### Detect the Anomaly Using Isolation Forest:

**Dynamically Setting the Threshold. **Once you have thousands of records, you cannot look at them one by one. You use statistical rules of thumb to isolate the threshold programmatically:


*   The Standard Deviation Rule (Extreme Outliers): Calculate the average score of your dataset. Flag any record that sits more than 2 or 3 standard deviations away from that average.
*   The Percentile Method for Auditing Budgets: If your quality assurance department only has the manpower to review 50 records a week, you simply sort your database by the lowest raw_anomaly_score and pull the Top 50 worst scores, regardless of percentage.








#### Statistical Thresholding Script

In [10]:
# Step 7: Try This Statistical Thresholding Script (Optimized for 1,000 Records)
import numpy as np
import textwrap
from sklearn.ensemble import IsolationForest
from google.colab import data_table

# 1. Initialize and run the Isolation Forest on your new 1000x768 vector matrix
print("Training Isolation Forest on production vector matrix...")
iso_forest_prod = IsolationForest(contamination='auto', random_state=42)
iso_forest_prod.fit(X)

# Extract raw anomaly scores (lower = more anomalous)
df['raw_anomaly_score'] = iso_forest_prod.score_samples(X)

# 2. Calculate statistical metrics
scores = df['raw_anomaly_score'].values
mean_score = np.mean(scores)
std_score = np.std(scores)

# Use a strict 2.5 standard deviation cutoff for a 1% anomaly rate
threshold = mean_score - (2.5 * std_score)

# Apply flags to the dataframe
df['dynamic_anomaly_flag'] = df['raw_anomaly_score'].apply(lambda x: "⚠️ ANOMALY" if x < threshold else "✅ Normal")

# 3. Print high-level summary metrics
print("\n==================================================")
print(f"📊 SYSTEM METRICS SUMMARY")
print("==================================================")
print(f"Calculated Dataset Mean Score: {mean_score:.4f}")
print(f"Statistical Anomaly Cutoff Threshold: {threshold:.4f}")

# 4. Explicitly list all flagged cases first with 80-character line wrapping
print("==================================================")
print("⚠️ FLAGGED ANOMALY ENCOUNTERS (FOR HUMAN REVIEW)")
print("==================================================")
anomalies_only = df[df['dynamic_anomaly_flag'] == "⚠️ ANOMALY"].sort_values(by='raw_anomaly_score')

print(f"Total anomalies flagged: {len(anomalies_only)}\n")

if len(anomalies_only) > 0:
    for index, row in anomalies_only.iterrows():
        print(f"• ID: {row['encounter_id']} | Specialty: {row['specialty']} | Score: {row['raw_anomaly_score']:.4f}")

        # Wrap text to 80 characters and indent subsequent lines cleanly
        wrapped_notes = textwrap.fill(row['clinical_notes'], width=80, initial_indent="  Notes: ", subsequent_indent="         ")
        print(f"{wrapped_notes}\n")
        print("-" * 40)
else:
    print("No anomalies detected under the current statistical threshold.\n")

# 5. Enable and display the entire dataset in a scrollable format
print("\n==================================================")
print("📋 COMPLETE ENCOUNTER DATASET (SCROLLABLE TABLE)")
print("==================================================")

# Configure Colab's interactive table output settings
data_table.enable_dataframe_formatter()
display(df[['encounter_id', 'specialty', 'raw_anomaly_score', 'dynamic_anomaly_flag']])


Training Isolation Forest on production vector matrix...

📊 SYSTEM METRICS SUMMARY
Calculated Dataset Mean Score: -0.4151
Statistical Anomaly Cutoff Threshold: -0.5253
⚠️ FLAGGED ANOMALY ENCOUNTERS (FOR HUMAN REVIEW)
Total anomalies flagged: 14

• ID: 1316 | Specialty: Gastroenterology | Score: -0.7085
  Notes: HISTORY OF PRESENT ILLNESS: ,  The patient is a 48-year-old man who has
         had abdominal pain since October of last year associated with a
         30-pound weight loss and then developed jaundice.  He had epigastric
         pain and was admitted to the hospital.  A thin-slice CT scan was
         performed, which revealed a 4 x 3 x 2 cm pancreatic mass with involved
         lymph nodes and ring enhancing lesions consistent with liver
         metastases.  The patient, additionally, had a questionable pseudocyst
         in the tail of the pancreas.  The patient underwent ERCP on 04/04/2007
         with placement of a stent.  This revealed a strictured pancreatic duct,


,encounter_id,specialty,raw_anomaly_score,dynamic_anomaly_flag
0,1001,Neurology,-0.402610,✅ Normal
1,1002,Pediatrics - Neonatal,-0.389251,✅ Normal
2,1003,Pediatrics - Neonatal,-0.397927,✅ Normal
3,1004,Pediatrics - Neonatal,-0.378008,✅ Normal
4,1005,Pediatrics - Neonatal,-0.400541,✅ Normal
...,...,...,...,...
515,2006,Neurology,-0.479218,✅ Normal
516,2007,Pediatrics,-0.467149,✅ Normal
517,2008,Cardiology,-0.488164,✅ Normal
518,2009,Gastroenterology,-0.473571,✅ Normal


In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()
df_anomalies = df[df['dynamic_anomaly_flag'] == "⚠️ ANOMALY"]
display(df_anomalies)

In [12]:
anomaly_counts_by_specialty = df_anomalies['specialty'].value_counts()
display(anomaly_counts_by_specialty)

,count
specialty,
Gastroenterology,7
Neurology,6
Pediatrics - Neonatal,1


#### Flag records as Anomalous.

In [13]:
# Step 8: This is a correction script, to suggest to which specialty a record should be assigned to
# in this case, for illustration, it uses the known value of the anomalous case (4), but could
# be modified to pull case record numbers whose values were based on thresholds.

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import textwrap

# Step 1: Isolate our known "good" clean records to build baseline profiles
# We dynamically exclude ANY record that our model flagged as an anomaly
clean_df = df[df['dynamic_anomaly_flag'] == "✅ Normal"]
anomalous_df = df[df['dynamic_anomaly_flag'] == "⚠️ ANOMALY"]

# Create a dictionary to store the baseline vector for each specialty
specialty_baselines = {}
for spec in clean_df['specialty'].unique():
    # Get all vectors belonging to this specialty
    spec_vectors = np.array(clean_df[clean_df['specialty'] == spec]['vector'].tolist())
    # Calculate the average vector profile for this specialty
    specialty_baselines[spec] = np.mean(spec_vectors, axis=0).reshape(1, -1)

# Step 2: Loop through and analyze every single detected anomaly
print("==================================================")
print("🧠 BATCH COGNSINE SIMILARITY ANALYSIS & ROUTING RECS")
print("==================================================")

if len(anomalous_df) == 0:
    print("No anomalies to analyze. Check your dynamic threshold settings.")
else:
    for _, row in anomalous_df.iterrows():
        print(f"\n🚨 RE-ROUTING PROFILE FOR ENCOUNTER ID: {row['encounter_id']}")
        print(f"Current Mislabeled Specialty: {row['specialty']}")

        # Format and wrap notes snippet for easier review
        wrapped_notes = textwrap.fill(row['clinical_notes'], width=80, initial_indent="Clinical Notes: ", subsequent_indent="                ")
        print(f"{wrapped_notes}\n")

        # Extract the vector for this specific anomaly
        anomaly_vector = row['vector'].reshape(1, -1)

        highest_score = -1
        recommended_specialty = None

        # Calculate similarity against all known baseline profiles
        print("Department Compatibility Scores:")
        for spec, baseline_vector in specialty_baselines.items():
            similarity_score = cosine_similarity(anomaly_vector, baseline_vector)[0][0]
            print(f"  • {spec.ljust(18)}: {similarity_score:.4f}")

            if similarity_score > highest_score:
                highest_score = similarity_score
                recommended_specialty = spec

        print("\n💡 SYSTEM RECOMMENDATION:")
        print(f"  Move to: **{recommended_specialty}** (Confidence Score: {highest_score:.4f})")
        print("-" * 50)


🧠 BATCH COGNSINE SIMILARITY ANALYSIS & ROUTING RECS

🚨 RE-ROUTING PROFILE FOR ENCOUNTER ID: 1066
Current Mislabeled Specialty: Pediatrics - Neonatal
Clinical Notes: PROCEDURE:, Delayed primary chest closure.,INDICATIONS: , The
                patient is a newborn with diagnosis of hypoplastic left heart
                syndrome who 48 hours prior to the current procedure has
                undergone a modified stage 1 Norwood operation.  Given the
                magnitude of the operation and the size of the patient (2.5 kg),
                we have elected to leave the chest open to facilitate
                postoperative management.  He is now taken back to the operative
                room for delayed primary chest closure.,PREOP DX: , Open chest
                status post modified stage 1 Norwood operation.,POSTOP DX:,
                Open chest status post modified stage 1 Norwood
                operation.,ANESTHESIA: , General endotracheal.,COMPLICATIONS: ,
                

In [ ]:
print("Encounter IDs of anomalous records:")
for encounter_id in anomalies_only['encounter_id']:
    print(encounter_id)

Open this window, and type "Resume medical anomaly project,"

https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfCORGHfCOoAuq7N4r7jnULryoQP9ma9pC4jQQXipsGY5RxDx0SoU3-_i6yEwlmdL02lpsDg5WHN79RU6d1S282tbqz_6-uzWKm5h9aoEYdfP06eb30U-uV41FDy0NOlRS5qIqTeaZ56PXp6UsOfDSXoXjG0GN1TmSqmO6nuaVsluOGtn9inSl004f0U0f-fpjbAjYYzyTf8daJsfBTjIgUu5T5CImeclcIlje7YHXWOl9rOozVfpK7t-elnPHzcIoA7Bkz3MEvI2A&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50

#### Make t-SNE plot:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# 1. Gather all of our vectors and reshape them into a standard 2D matrix
embeddings_matrix = np.array(df['vector'].tolist())

# 2. Configure t-SNE to compress our 768 dimensions into 2 dimensions (X, Y)
# Note: perplexity is set low due to our sample size, adjusting it guides neighborhood density
tsne = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings_matrix)

# Map the 2D coordinates back onto our main dataframe for simple plotting
df['tsne_x'] = embeddings_2d[:, 0]
df['tsne_y'] = embeddings_2d[:, 1]

# 3. Setup the visual plot canvas
plt.figure(figsize=(12, 8))
plt.title("Medical Record Vector Space Map (t-SNE Clusters)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("t-SNE Dimension 1", fontsize=10)
plt.ylabel("t-SNE Dimension 2", fontsize=10)

# Define clean colors for our 5 standard specialties
specialty_colors = {
    'Cardiology': '#1f77b4',       # Muted Blue
    'Orthopedics': '#2ca02c',      # Muted Green
    'Pediatrics': '#9467bd',       # Muted Purple
    'Neurology': '#ff7f0e',        # Muted Orange
    'Gastroenterology': '#bcbd22'  # Muted Olive
}

# 4. First, plot the clean ("Normal") records as standard colored circular dots
for spec, color in specialty_colors.items():
    normal_subset = df[(df['specialty'] == spec) & (df['dynamic_anomaly_flag'] == "✅ Normal")]
    plt.scatter(
        normal_subset['tsne_x'],
        normal_subset['tsne_y'],
        label=f"{spec} (Normal)",
        color=color,
        alpha=0.6,
        s=60,
        edgecolors='none'
    )

# 5. Overplot all dynamically flagged anomalies as stark, large red triangles
anomalies_subset = df[df['dynamic_anomaly_flag'] == "⚠️ ANOMALY"]

if len(anomalies_subset) > 0:
    plt.scatter(
        anomalies_subset['tsne_x'],
        anomalies_subset['tsne_y'],
        label="⚠️ Detected Anomaly",
        color='#d62728',  # Crisp Crimson Red
        marker='^',       # Triangle Marker
        s=150,            # Larger size to pop visually
        edgecolors='black',
        linewidths=1.5,
        zorder=5          # Ensures they stay layered on top
    )

    # Label each anomaly triangle with its unique Encounter ID
    for _, row in anomalies_subset.iterrows():
        plt.annotate(
            f"ID: {row['encounter_id']}",
            (row['tsne_x'], row['tsne_y']),
            textcoords="offset points",
            xytext=(0, 10),
            ha='center',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.6, ec="orange")
        )

# Clean up layout, position legend nicely outside data field, and add grid alignment
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
print("==================================================")
print("📥 RECORDS CAUGHT BY PRE-EMBEDDING AUDIT FILTERS")
print("==================================================")

# 1. Print out the Under-Documentation Filter captures (Queue A)
print(f"⚠️ QUEUE A: Insufficient Text / Blank Records ({len(queue_a)} total)")
if len(queue_a) > 0:
    for _, item in queue_a.iterrows():
        print(f"  • ID: {item['encounter_id']} | Specialty: {item['specialty']}")
else:
    print("  No records caught.")

print("\n" + "-"*50 + "\n")

# 2. Print out the Mid-Sentence Save Drop captures (Queue B)
print(f"⚠️ QUEUE B: Mid-Sentence Save Drops ({len(queue_b)} total)")
if len(queue_b) > 0:
    for _, item in queue_b.iterrows():
        print(f"  • ID: {item['encounter_id']} | Specialty: {item['specialty']}")
else:
    print("  No records caught.")
print("==================================================")